# Linear regression — PDMP sampler comparison

Small linear regression (N=200, D=8, K=3 signals). Loads pre-saved results from disk; no samplers are re-run.

In [ ]:
%matplotlib inline
import os, sys, json
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

BASE_DIR = Path("results/linear_regression")
PLI_DIR    = BASE_DIR / "pli"    / "N200_D8_K3_seed0"
BRENT_DIR  = BASE_DIR / "brent"  / "N200_D8_K3_seed0"

STYLE = {
    "Analytic":    ("k",  "D"),
    "NUTS":        ("C2", "D"),
    "NUTS-HS":     ("C4", "P"),
    "Boom":        ("C0", "o"),
    "Sticky-Boom": ("C1", "s"),
    "ZZ":          ("C9", "v"),
    "Sticky-ZZ":   ("C3", "^"),
}

STICKY_NAMES = {"Sticky-Boom", "Sticky-ZZ"}

def load_dir(results_dir: Path, thinning_label: str):
    rows = []
    if not results_dir.exists():
        print(f"WARNING: {results_dir} not found — skipping {thinning_label}")
        return rows
    for pt_file in sorted(results_dir.glob("*.pt")):
        d = torch.load(pt_file, weights_only=False)
        name = d["name"]
        color, marker = STYLE.get(name, ("grey", "o"))
        rows.append({
            "name":     name,
            "thinning": thinning_label,
            "label":    name if thinning_label == "PLI" else f"{name} (Brent)",
            "samples":  d["samples"].numpy() if hasattr(d["samples"], "numpy") else np.array(d["samples"]),
            "wall":     d["wall"],
            "config":   d["config"],
            "color":    color,
            "marker":   marker,
            "sticky":   name in STICKY_NAMES,
        })
    return rows

pli_rows   = load_dir(PLI_DIR,   "PLI")
brent_rows = load_dir(BRENT_DIR, "Brent")

# Shared methods (Analytic, NUTS, NUTS-HS) appear in PLI only — no need to duplicate
shared_names = {"Analytic", "NUTS", "NUTS-HS"}
brent_rows_only = [r for r in brent_rows if r["name"] not in shared_names]

# All rows for full comparison; order: PLI first, then Brent-only extras
order = list(STYLE.keys())
pli_rows.sort(key=lambda r: order.index(r["name"]) if r["name"] in order else 99)
brent_rows_only.sort(key=lambda r: order.index(r["name"]) if r["name"] in order else 99)

rows = pli_rows + brent_rows_only

# Separate views for thinning-specific comparisons
pdmp_names = {"Boom", "Sticky-Boom", "ZZ", "Sticky-ZZ"}
pli_pdmp   = [r for r in pli_rows   if r["name"] in pdmp_names]
brent_pdmp = [r for r in brent_rows if r["name"] in pdmp_names]

# Load metrics from whichever dir is available (prefer PLI for shared ground truth)
_meta_dir = PLI_DIR if PLI_DIR.exists() else BRENT_DIR
with open(_meta_dir / "metrics.json") as f:
    meta = json.load(f)

cfg        = rows[0]["config"]
true_coefs = np.array(meta["true_coefs"])
is_signal  = np.array(meta["is_signal"], dtype=bool)
coef_names = ["intercept"] + [f"β_{i}" for i in range(cfg["D"])]

print(f"PLI methods:   {[r['name'] for r in pli_rows]}")
print(f"Brent methods: {[r['name'] for r in brent_rows]}")
print(f"true_coefs = {np.round(true_coefs, 3)}")
print(f"is_signal  = {is_signal.tolist()}")

## Samplers

Two families of samplers are compared:

- **Non-sticky** (Analytic, NUTS, NUTS-HS, Boom, ZZ): target the standard Gaussian posterior p(β | X, y).
- **Sticky** (Sticky-Boom, Sticky-ZZ): target a *mixed measure* with atomic mass at zero on each coordinate — a spike-and-slab posterior where coordinates can be exactly zero with positive probability.

Comparing sticky marginals to the analytic Gaussian is not comparing the same two objects.

## Bayesian metrics table

In [ ]:
N, D = cfg["N"], cfg["D"]
lik_noise_std = cfg["lik_noise_std"]

test_rng   = np.random.default_rng(cfg["seed"] + 1)
X_te_raw   = test_rng.normal(size=(N, D))
X_te       = (X_te_raw - X_te_raw.mean(0)) / X_te_raw.std(0)
y_te_clean = X_te @ true_coefs[1:] + true_coefs[0]
y_te_noisy = y_te_clean + cfg["noise_std"] * test_rng.normal(size=N)
X_te_aug   = np.column_stack([np.ones(N), X_te])

an_row = next((r for r in pli_rows if r["name"] == "Analytic"), None)
std_an = an_row["samples"].std(0) if an_row is not None else None

def compute_elpd(samples, X_te_aug, y_te_noisy, sigma):
    mu_pred  = X_te_aug @ samples.T
    log_liks = stats.norm.logpdf(y_te_noisy[:, None], loc=mu_pred, scale=sigma)
    log_ml   = np.log(np.mean(np.exp(log_liks - log_liks.max(1, keepdims=True)), axis=1)) + log_liks.max(1)
    return float(log_ml.mean())

table_rows = []
for r in rows:
    s    = r["samples"]
    wall = r["wall"]

    pmean     = s.mean(0)
    rmse_mean = float(np.sqrt(np.mean((pmean - true_coefs) ** 2)))

    if std_an is not None:
        std_ratio = float(np.mean(s[:, is_signal].std(0) / std_an[is_signal]))
    else:
        std_ratio = float("nan")

    elpd      = compute_elpd(s, X_te_aug, y_te_noisy, lik_noise_std)
    pred_mean = X_te_aug @ pmean
    pred_rmse = float(np.sqrt(np.mean((pred_mean - y_te_clean) ** 2)))

    if r["sticky"]:
        null_mask = ~is_signal
        p0_null   = float(np.mean(np.mean(np.abs(s[:, null_mask]) < 1e-8, axis=0)))
        p0_sig    = float(np.mean(np.mean(np.abs(s[:, is_signal]) < 1e-8, axis=0)))
    else:
        p0_null = float("nan")
        p0_sig  = float("nan")

    table_rows.append({
        "sampler":        r["label"],
        "thinning":       r["thinning"],
        "n_draws":        len(s),
        "wall_sec":       round(wall, 2) if wall is not None else None,
        "post_mean_rmse": round(rmse_mean, 4),
        "post_std_ratio": round(std_ratio, 3),
        "elpd":           round(elpd, 4),
        "pred_rmse":      round(pred_rmse, 4),
        "p0_null":        round(p0_null, 3) if not np.isnan(p0_null) else None,
        "p0_signal":      round(p0_sig, 3)  if not np.isnan(p0_sig)  else None,
    })

df = pd.DataFrame(table_rows).set_index(["thinning", "sampler"])
df

## Coefficient calibration plot

One subplot per coefficient. Violin for non-sticky samplers; for sticky samplers a two-tone violin to emphasise the spike at zero. True value is the red dashed line. Yellow background = true signal coordinate.

In [ ]:
n_coords = len(true_coefs)

nrows, ncols = 3, 3
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.8 * nrows), sharey=False)
axes = axes.flatten()

offsets = np.linspace(-0.35, 0.35, len(rows))

for ax_idx, ax in enumerate(axes):
    if ax_idx >= n_coords:
        ax.axis("off")
        continue

    coord = ax_idx
    if is_signal[coord]:
        ax.set_facecolor("#fffbe6")

    ax.axhline(true_coefs[coord], color="crimson", lw=1.5, ls="--", zorder=5)

    vp = ax.violinplot(
        [r["samples"][:, coord] for r in rows],
        positions=list(range(len(rows))),
        widths=0.7,
        showmedians=True,
        showextrema=False,
    )

    for body, r in zip(vp["bodies"], rows):
        body.set_facecolor(r["color"])
        body.set_alpha(0.55 if r["thinning"] == "PLI" else 0.30)
        if r["thinning"] == "Brent":
            body.set_hatch("//")
            body.set_edgecolor(r["color"])

    vp["cmedians"].set_color("k")
    vp["cmedians"].set_linewidth(1.2)

    for pos, r in zip(range(len(rows)), rows):
        if r["sticky"]:
            s  = r["samples"][:, coord]
            p0 = np.mean(np.abs(s) < 1e-8)
            ax.text(pos, ax.get_ylim()[0] if ax.get_ylim()[0] < 0 else 0,
                    f"p₀={p0:.2f}", ha="center", va="bottom",
                    fontsize=5.5, color=r["color"], fontweight="bold")

    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels([r["label"] for r in rows], rotation=90, fontsize=6.5)
    ax.set_title(coef_names[coord], fontsize=8)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

handles = []
for r in rows:
    patch = mpatches.Patch(facecolor=r["color"], label=r["label"],
                           alpha=0.55 if r["thinning"] == "PLI" else 0.30,
                           hatch="//" if r["thinning"] == "Brent" else "")
    handles.append(patch)

fig.legend(handles=handles, loc="upper center", ncol=len(rows),
           fontsize=7, frameon=False, bbox_to_anchor=(0.5, 1.01))
fig.suptitle("Posterior marginals per coefficient  (red dashed = true value, hatch = Brent thinning)",
             fontsize=9, y=1.03)
plt.tight_layout()
plt.show()

## Conditional posteriors (sticky only)

For each sticky sampler and each **signal** coordinate: histogram of samples conditioned on β ≠ 0, overlaid with the analytic Gaussian marginal. Tests whether the slab component of the sticky posterior matches the true Gaussian posterior.

In [ ]:
all_sticky = [r for r in rows if r["sticky"]]
sig_coords = np.where(is_signal)[0]

if an_row is None:
    print("No Analytic sampler found — skipping conditional posteriors.")
else:
    an_samples = an_row["samples"]
    n_sticky   = len(all_sticky)
    n_sig      = len(sig_coords)

    fig, axes = plt.subplots(n_sticky, n_sig,
                             figsize=(3.5 * n_sig, 3.0 * n_sticky),
                             squeeze=False)

    for row_idx, r in enumerate(all_sticky):
        for col_idx, coord in enumerate(sig_coords):
            ax = axes[row_idx][col_idx]

            s_all   = r["samples"][:, coord]
            nonzero = s_all[np.abs(s_all) > 1e-8]
            frac_nz = len(nonzero) / len(s_all)

            an_c  = an_samples[:, coord]
            mu_an = an_c.mean()
            sd_an = an_c.std()

            grid = np.linspace(
                min(an_c.min(), nonzero.min() if len(nonzero) else an_c.min()),
                max(an_c.max(), nonzero.max() if len(nonzero) else an_c.max()),
                300,
            )
            ax.plot(grid, stats.norm.pdf(grid, mu_an, sd_an),
                    color="k", lw=1.5, label="Analytic N(μ,σ²)")

            if len(nonzero) >= 10:
                ax.hist(nonzero, bins=40, density=True,
                        color=r["color"], alpha=0.5,
                        hatch="//" if r["thinning"] == "Brent" else "",
                        label=f"| β≠0  ({frac_nz:.0%})")
            else:
                ax.text(0.5, 0.5, "all zero", transform=ax.transAxes,
                        ha="center", va="center", fontsize=9)

            ax.axvline(true_coefs[coord], color="crimson", lw=1.2, ls="--",
                       label=f"true={true_coefs[coord]:.2f}")
            ax.set_title(f"{r['label']} — {coef_names[coord]}", fontsize=8)
            ax.legend(fontsize=6.5, frameon=False)
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)

    fig.suptitle("Conditional posterior (given β ≠ 0) vs analytic Gaussian", fontsize=10)
    plt.tight_layout()
    plt.show()

## ESS table

Geyer IPS ESS for non-sticky samplers. For sticky samplers: **active ESS** (ESS of non-zero draws per coord) and **indicator ESS** (ESS of 1[β≠0]).

In [ ]:
def ess_ips(samples: np.ndarray, max_lag: int = 1000) -> np.ndarray:
    """Geyer initial-positive-sequence ESS, vectorised over coordinates."""
    x = samples - samples.mean(0, keepdims=True)
    n, d = x.shape
    var = (x ** 2).mean(0)
    rho_sum   = np.zeros(d)
    prev_pair = np.full(d, np.inf)
    active    = np.ones(d, dtype=bool)
    k = 1
    while k + 1 <= min(max_lag, n - 2):
        c_k   = (x[:n - k]     * x[k:]).mean(0)     / np.maximum(var, 1e-30)
        c_kp1 = (x[:n - k - 1] * x[k + 1:]).mean(0) / np.maximum(var, 1e-30)
        pair  = c_k + c_kp1
        kill  = active & ((pair <= 0) | (pair >= prev_pair))
        active = active & ~kill
        rho_sum += np.where(active, pair, 0.0)
        prev_pair = np.where(active, pair, prev_pair)
        if not active.any():
            break
        k += 2
    tau = 1.0 + 2.0 * rho_sum
    return n / np.clip(tau, 1.0, None)


ess_table = []

for r in rows:
    s    = r["samples"]
    wall = r["wall"]

    if r["sticky"]:
        nonzero_mask = np.abs(s) > 1e-8

        ess_active = np.full(s.shape[1], np.nan)
        for coord in range(s.shape[1]):
            sub = s[nonzero_mask[:, coord], coord]
            if len(sub) >= 50:
                ess_active[coord] = ess_ips(sub[:, None]).item()

        ess_ind = ess_ips(nonzero_mask.astype(float))

        for label, ess in [("active", ess_active), ("indicator", ess_ind)]:
            mn, md = float(np.nanmin(ess)), float(np.nanmedian(ess))
            ess_table.append({
                "sampler":    r["label"],
                "thinning":   r["thinning"],
                "metric":     label,
                "min_ESS":    round(mn),
                "med_ESS":    round(md),
                "min_ESS/sec": round(mn / wall, 1) if wall else None,
            })
    else:
        ess = ess_ips(s)
        mn, md = float(np.min(ess)), float(np.median(ess))
        ess_table.append({
            "sampler":    r["label"],
            "thinning":   r["thinning"],
            "metric":     "ESS",
            "min_ESS":    round(mn),
            "med_ESS":    round(md),
            "min_ESS/sec": round(mn / wall, 1) if wall else None,
        })

ess_df = pd.DataFrame(ess_table).set_index(["thinning", "sampler", "metric"])
ess_df

## Prediction interval calibration

For each nominal coverage level α ∈ {50%, 80%, 90%, 95%}: compute empirical coverage of the equal-tailed predictive interval across test points. Predictive samples are drawn from N(x*ᵀβₛ, σ²_lik) for each posterior draw βₛ.

In [ ]:
nominal_levels = np.array([0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95])

calib_rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot([0, 1], [0, 1], "k--", lw=0.9, label="perfect calibration")

for r in rows:
    s       = r["samples"]
    mu_pred = X_te_aug @ s.T
    noise   = calib_rng.normal(scale=lik_noise_std, size=mu_pred.shape)
    y_pred  = mu_pred + noise

    empirical = []
    for alpha in nominal_levels:
        lo  = np.quantile(y_pred, (1 - alpha) / 2, axis=1)
        hi  = np.quantile(y_pred, (1 + alpha) / 2, axis=1)
        cov = float(np.mean((y_te_noisy >= lo) & (y_te_noisy <= hi)))
        empirical.append(cov)

    ls = "--" if r["thinning"] == "Brent" else "-"
    ax.plot(nominal_levels, empirical, marker=r["marker"], color=r["color"],
            lw=1.5, ls=ls, label=r["label"], markersize=6)

ax.set_xlabel("Nominal coverage")
ax.set_ylabel("Empirical coverage")
ax.set_title("Prediction interval calibration\n(dashed = Brent thinning)")
ax.set_xticks(nominal_levels)
ax.set_xticklabels([f"{int(l*100)}%" for l in nominal_levels])
ax.legend(fontsize=7.5, frameon=False)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()